# 04 · Evaluation

| Stage | Notebook |
|---|---|
| 1. Data quality & EDA | `01_eda.ipynb` |
| 2. Feature engineering | `02_feature_engineering.ipynb` |
| 3. Model training | `03_model_training.ipynb` |
| **4. Evaluation** | **this notebook** |

**Inputs** (written by `python scripts/run_phase1.py`, the single source of truth):
* `results/predictions.csv.gz`: raw scores of every model on every flight, plus each outcome's availability date. Calibration (including rolling recalibration), metrics, policies, drift and the bootstrap are **recomputed here** with the same `aerocast.pipeline` functions, and **each table is asserted equal to its saved artefact** (`artefacts.assert_matches`).
* Ablation, permutation importance, gain/coefficients, target sensitivity, PSI and adversarial validation need refitted models, so they are **loaded** from `results/` (a loaded table is the artefact itself).

**How this notebook is organised.** Evidence is kept apart by what it may be used for:

| Part | Content | May it drive a decision? |
|---|---|---|
| **A. Evidence used for selection (CV / Sep–Oct)** | rolling-CV model comparison, history tuning, month check, ablation CV gains, the calibration rule on Sep–Oct, importance on Sep–Oct | yes: these are the only numbers any design choice rests on |
| **B. Exploratory test-set findings** | test metrics and bootstrap, policies, calibration study, ablation test differences, target sensitivity, covariate shift | **no**: reported, not used to choose anything. See the Development protocol (§0) for what was seen and when |

Every **reference row** (the *prior* and the *route-history* heuristic) sits beside the learned models, and every learned row names its **calibration variant**.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parent / "src"))  # make `aerocast` importable
import warnings; warnings.filterwarnings("ignore")
import json
import numpy as np, pandas as pd
from IPython.display import display
from aerocast import artefacts, config, evaluate, pipeline, plots
pd.set_option("display.precision", 4); pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_columns", 40)
R = config.RESULTS_DIR
S = json.loads((R / "summary.json").read_text())
FINAL = config.FINAL_MODEL

labels, raw_preds = pipeline.load_predictions()
print("final model (fixed before test):", S["final_model"]["variant"], "| model family:", S["final_model"]["model"])

final model (fixed before test): LogReg (Platt, rule-selected) | model family: logreg


<a id="protocol"></a>
## 0. Development protocol (disclosure)

The test window (Nov–Dec) is meant to be touched once. Here is an honest list of what was seen during development, and how each decision now rests on **CV / Sep–Oct evidence only**.

**Diagnostics that printed test metrics before the design was frozen:**
1. **First run** (single-September tuning design): test PR-AUC and calibration of all models were printed. The design was abandoned because of the September evidence (XGBoost stopped at 13 trees), but test numbers had been seen.
2. **Month features:** a with/without comparison printed test PR-AUC, and that was the original justification for removing them. **Withdrawn.** Re-justified by rolling CV (notebook 02, §7).
3. **Feature-subset variants** (dropping log-counts, IDs, day-level history, congestion; a minimal set) printed calibration-window *and* test PR-AUC and test recall@precision. These shaped the "day-level features memorise days" hypothesis. **Re-justified** by the distinct-value analysis and the CV ablation (notebook 02, §8; Part A.4 below).
4. **Precision target:** 0.50 → 0.40 was decided because 0.50 is unreachable on Sep–Oct (base rate 15.7%), but the same diagnostic also printed test recall at 0.35/0.40/0.50.
5. **Calibration:** test ECE of raw vs Platt vs isotonic was seen *before* any selection rule existed. The "fit Sep → check Oct" rule, the shift-aware Platt variant (evaluated on test ECE), and the 0.005-margin rule were all introduced **after** that.
6. **First review round:** the full pipeline (which also computes test metrics) ran once while the final-model comparison still used XGBoost's optimistic in-fold early-stopping score. The comparison was corrected for a methodological reason identified from CV alone (notebook 03, §7). That correction reversed the choice to LogReg.
7. **Reference baselines in the policy table:** adding the *prior* to the policy comparison showed an apparent lift on both Sep–Oct and test for a constant score. The cause was an order-based tie-break in the daily top-k% policy (rows are in departure order, so ties picked the day's latest flights). The tie-break is now seeded random. This is an evaluation-hygiene fix. It leaves model scores, the policy definition and the share (10%) unchanged, and it only alters the daily-policy numbers of scores that contain exact ties (the prior, and route history to a small extent).
8. **This round (items 1–11 of the second review):** labels, baselines, assertions and documentation only. No modelling decision changed. The joint re-tuning of (m, window, C) re-selected the same values (Part A.2).

**Decided without test evidence:** the decision clock (and clock vs 2-day lag, CV), smoothing m and window (CV, jointly with C), C and XGBoost configuration (CV), final model (CV as deployed), calibration margin rule and its outcome (Sep→Oct), rolling-recalibration window N = 28 (a priori), daily top-10% policy (a priori capacity assumption), and the precision target (Sep–Oct).

**Implication.** Test numbers are not perfectly clean confirmations. The design was iterated with test results visible on several occasions, so treat small test differences with caution. Every conclusion below is stated with its CV or Sep–Oct basis, and Part B is labelled exploratory for that reason.

## 1. Metric theory in brief

* **PR-AUC (average precision).** sklearn's `average_precision_score` computes $AP = \sum_k (R_k - R_{k-1})\,P_k$, a step-wise sum over thresholds. It is *not* the trapezoidal area under the PR curve, which interpolates linearly between points and is optimistic because PR curves aren't convex. "PR-AUC" in the tables means AP. Its no-skill baseline is the **base rate** (the *prior* row).
* **PR vs ROC under imbalance.** ROC-AUC is the probability that a random late flight is ranked above a random on-time flight. It ignores the base rate, so the same ROC can mean very different precision when positives are rare. PR curves put precision on the axis, which is what an operations desk experiences. Both are reported.
* **ECE** (Guo et al., 2017): $\mathrm{ECE} = \sum_b \frac{|B_b|}{n}\,\big|\bar y_b - \bar p_b\big|$ over bins of predicted probability.
  * It's a **biased estimator**: sampling noise in each bin's $\bar y_b$ always adds, so ECE > 0 even for a perfectly calibrated model in finite samples, and the bias grows with the number of bins.
  * Results also depend on the binning. **Equal-width** bins (15) leave the sparse high-risk tail with few flights; **equal-mass** bins (quantiles, `ece_equal_mass`) give every bin the same count. Both are reported, and the Brier score (proper, binning-free) is reported alongside.
  * ECE is a **marginal** check: the *prior* (a constant equal to the training base rate) has a small ECE whenever the test base rate happens to be near the training one. A low ECE therefore says nothing about ranking skill, which is why ECE is always read together with PR-AUC and Brier.
* **Platt / temperature scaling vs isotonic.** Platt fits $\sigma(a\,\mathrm{logit}(p)+b)$: 2 parameters, monotone, smooth, data-efficient. Temperature scaling is its 1-parameter special case ($b=0$), the multi-class tool in Guo et al. Isotonic regression fits any monotone step function: more flexible, but needs more data and can overfit. All of them assume the calibration data are **distributed like deployment data**.
* **`oracle_recall@p`.** The best recall at precision ≥ p, with the threshold **tuned on the evaluated labels themselves**. It is a point on the PR curve, a **model-comparison metric only**, never an operating result (no deployable threshold produces it). The deployable results are the two policies in Part B.
* **Why the bootstrap resamples whole days.** Flights on the same day share weather, air-traffic programmes and cascading delays, so their errors are correlated. Resampling individual flights treats about 900 correlated flights as independent and gives intervals that are far too narrow. Resampling **days** keeps each day's dependence intact (a block bootstrap). Model differences are computed on the *same* resampled days (paired).

# Part A · Evidence used for selection (CV / Sep–Oct)

*Everything below uses the training window (rolling-origin CV) or the calibration window (Sep–Oct). No test labels. Linked to the Development protocol: [§0](#protocol).*

## A.1 Final-model selection: rolling-CV PR-AUC, as deployed

In [2]:
cv_tbl = pd.read_csv(R / "cv_summary.csv")
cv_tbl.drop(columns="variant").round(4)

,model,calibration,cv_pr_auc_mean,cv_pr_auc_std,fold_m5,fold_m6,fold_m7,fold_m8
0,Prior (base rate),none,0.2775,0.0483,0.2267,0.3152,0.3221,0.2460
1,Route history,none,0.3476,0.0601,0.2766,0.3911,0.4034,0.3194
2,LogReg (as deployed),raw,0.4705,0.0706,0.4354,0.5029,0.5515,0.3922
3,"XGBoost (as deployed, fixed trees)",raw,0.4450,0.0729,0.4044,0.4284,0.5523,0.3950
4,"XGBoost (in-fold early stopping, optimistic; not used)",raw,0.4754,0.0673,0.4490,0.4956,0.5574,0.3995


In [3]:
pair = S["cv_paired_logreg_minus_xgboost"]
pd.Series({"per-fold LogReg − XGBoost (May..Aug)": np.round(pair["diffs"], 4).tolist(), "mean": pair["mean"],
           "sd": pair["std"], "folds LogReg better": f'{pair["folds_a_better"]}/{pair["n_folds"]}'}, name="paired").to_frame()

,paired
per-fold LogReg − XGBoost (May..Aug),"[0.031, 0.0746, -0.0007, -0.0027]"
mean,0.0255
sd,0.0362
folds LogReg better,2/4


**LogReg and XGBoost are not statistically distinguished.** In CV, as deployed: LogReg 0.4705 vs XGBoost 0.4450, with a fold sd of about 0.07 for each. The test bootstrap in Part B also gives a CI for XGBoost − LogReg that includes 0 ([-0.010, +0.041]). The **final-model choice rests on the rule declared before test**: the model with the higher CV mean as deployed (LogReg), recorded in `config.FINAL_MODEL`. It does not rest on a demonstrated advantage. Both learned models clearly beat the references: the route-history heuristic scores 0.3476 and the prior 0.2775 in CV.

## A.2 Feature-design evidence (CV): history settings, decision clock, month

**Smoothing strength m, long window, and C are tuned jointly.** Every (m, window) pair is scored at its own best C by rolling-origin CV (LogReg). The clock rows repeat the configured pair with the clock moved to 00:00 on D-1 (≈ a 2-day lag) and to 24:00 on D-1 (every D-1 arrival counts for D: the leaky variant).

In [4]:
tuning = pd.read_csv(R / "history_tuning.csv")
tuning.round(4)

,smoothing_m,long_window,decision_hour,best_C,fold_m5,fold_m6,fold_m7,fold_m8,cv_pr_auc_mean,cv_pr_auc_std,cv_pr_auc_mean_C0.001
0,5,14,20,0.100,0.4354,0.5029,0.5515,0.3922,0.4705,0.0706,0.4684
1,5,28,20,0.001,0.4393,0.4952,0.5397,0.3925,0.4667,0.0643,0.4667
2,5,56,20,0.010,0.4351,0.5019,0.5441,0.3900,0.4678,0.0686,0.4662
3,20,14,20,0.010,0.4314,0.5032,0.5496,0.3916,0.4689,0.0709,0.4669
4,20,28,20,0.001,0.4369,0.4942,0.5389,0.3929,0.4657,0.0640,0.4657
5,20,56,20,0.010,0.4322,0.5011,0.5431,0.3897,0.4665,0.0686,0.4648
6,50,14,20,0.010,0.4244,0.4999,0.5469,0.3879,0.4648,0.0719,0.4631
7,50,28,20,0.001,0.4327,0.4915,0.5369,0.3914,0.4631,0.0641,0.4631
8,50,56,20,0.010,0.4287,0.4984,0.5413,0.3882,0.4641,0.0686,0.4626
9,5,14,0,0.003,0.4149,0.4971,0.5407,0.3821,0.4587,0.0730,0.4584


In [5]:
grid = tuning[tuning.decision_hour == config.DECISION_HOUR].pivot(index="smoothing_m", columns="long_window", values="cv_pr_auc_mean")
grid.round(4)

long_window,14,28,56
smoothing_m,,,
5,0.4705,0.4667,0.4678
20,0.4689,0.4657,0.4665
50,0.4648,0.4631,0.4641


* **The re-tuning is unchanged.** With C tuned jointly, the CV-best pair is still **m = 5, window = 14 days** (best C = 0.1), the configured values. The earlier fixed-C = 0.001 score (column `cv_pr_auc_mean_C0.001`) ranks the grid the same way at the top. The pipeline fails if the config ever disagrees with this table.
* **Decision clock:** the 20:00 clock scores 0.4705 vs 0.4587 for a 2-day lag (better in 4 of 4 folds). Counting every D-1 arrival (24:00, leaky) would score 0.4812, i.e. 0.011 higher, because it lets late-evening D-1 outcomes into D's features. That difference is the size of the leak the clock removes.

In [6]:
mcv = pd.read_csv(R / "month_feature_cv.csv")
mcv.round(4)

,features,model,fold_m5,fold_m6,fold_m7,fold_m8,cv_pr_auc_mean
0,without month,logreg,0.4354,0.5029,0.5515,0.3922,0.4705
1,without month,xgboost,0.4044,0.4284,0.5523,0.3950,0.4450
2,with month,logreg,0.4265,0.5050,0.5490,0.3938,0.4686
3,with month,xgboost,0.4112,0.4257,0.5633,0.3882,0.4471


* **Month features:** adding cyclic month features **does not help the final model** (LogReg 0.4705 → 0.4686, worse in 2 of 4 folds). For XGBoost the difference is +0.0021 (0.4450 → 0.4471). The structural argument: training covers months 1–8, so months 9–12 would be an extrapolation. **Decision: no month feature.** (The earlier test-based justification is withdrawn.)

## A.3 Calibration rule, decided on Sep–Oct

**Static rule (fixed before test, `config.CALIB_MIN_ECE_GAIN = 0.005`):** fit Platt and isotonic on **September**, measure ECE on **October**, and keep the **raw** scores unless the better recalibrator lowers October ECE by more than 0.005. The winner is then refit on all of Sep–Oct. This replaces the earlier "lowest October ECE wins" rule, which picked Platt for LogReg on a 0.0007 near-tie.

In [7]:
final_preds, choice, calib_tbl = pipeline.calibrate(labels, raw_preds)
print(artefacts.assert_matches(calib_tbl, "calibrator_selection.csv"))
print("rule-selected variants:", {m: pipeline.variant_label(f"{m}_rule_selected", choice)[0] for m in pipeline.MODELS})
calib_tbl.set_index(["model", "calibrator"]).round(4)

OK: identical to results/calibrator_selection.csv
rule-selected variants: {'logreg': 'LogReg (Platt, rule-selected)', 'xgboost': 'XGBoost (raw, rule-selected)'}


check_ece  check_ece_equal_mass  check_brier  gain_vs_raw
model   calibrator                                                           
logreg  raw            0.0609                0.0614       0.1404       0.0000
        platt          0.0349                0.0349       0.1378       0.0260
        isotonic       0.0362                0.0356       0.1386       0.0247
xgboost raw            0.0220                0.0245       0.1386       0.0000
        platt          0.0465                0.0465       0.1411      -0.0245
        isotonic       0.0468                0.0468       0.1413      -0.0248

* **LogReg → Platt (rule-selected).** Raw LogReg over-predicts in October (it carries the 26.5% training base rate into a month with a lower rate). Platt improves October ECE by 0.026, well above the 0.005 margin, so the rule clearly adopts it.
* **XGBoost → raw (rule-selected).** Recalibration does not beat raw by the margin.
* The causal **rolling Platt** (N = 28, fixed a priori) is the adaptive alternative. It is evaluated in Part B, and its N was **not** chosen on test (14 and 56 are reported as sensitivity only).

## A.4 Feature-family ablation: the CV evidence

In [8]:
steps = pd.read_csv(R / "ablation_steps.csv")
steps[["model", "calibration", "step", "cv_gain_mean", "cv_folds_improved"]].round(4)

,model,calibration,step,cv_gain_mean,cv_folds_improved
0,LogReg,raw,+ congestion,0.0209,4/4
1,LogReg,raw,+ history_entity,0.0195,4/4
2,LogReg,raw,+ history_day,0.0110,3/4
3,XGBoost,raw,+ congestion,0.0118,3/4
4,XGBoost,raw,+ history_entity,-0.0019,2/4
5,XGBoost,raw,+ history_day,-0.0040,3/4


Each step adds one family to the previous set, refit with the CV-chosen hyperparameters (raw scores). **Claims rest on these CV gains** (mean gain and the number of the 4 folds that improve):
* **LogReg (raw), the final model:** congestion +0.021 (4/4 folds), entity history +0.020 (4/4 folds), day-level history +0.011 (3/4 folds).
* **XGBoost (raw):** congestion +0.012 (3/4 folds), entity history -0.002 (2/4 folds), day-level history -0.004 (3/4 folds). CV and test disagree for XGBoost on some steps (Part B), so no claim is based on its ablation.

The ablation table with the **reference rows** (prior and route history are scored on the same folds):

In [9]:
abl = pd.read_csv(R / "ablation.csv")
abl[["model", "calibration", "features", "n_features", "cv_pr_auc_mean", "cv_pr_auc_std"]].round(4)

,model,calibration,features,n_features,cv_pr_auc_mean,cv_pr_auc_std
0,Prior (base rate),none,(baseline),0,0.2775,0.0483
1,Route history,none (smoothed rate),(baseline),0,0.3476,0.0601
2,LogReg,raw,schedule + ids,13,0.4191,0.0643
3,LogReg,raw,+ congestion,20,0.4400,0.0729
4,LogReg,raw,+ history_entity,38,0.4595,0.0727
5,LogReg,raw,+ history_day,45,0.4705,0.0706
6,LogReg,raw,history only,25,0.4478,0.0753
7,XGBoost,raw,schedule + ids,13,0.4391,0.0754
8,XGBoost,raw,+ congestion,20,0.4509,0.0886
9,XGBoost,raw,+ history_entity,38,0.4490,0.0769


In [10]:
figs = {}
figs["ablation_cv"] = plots.ablation_bars(abl); plots.save(figs["ablation_cv"], "ablation_cv"); figs["ablation_cv"]

<Figure size 650x370 with 1 Axes>

## A.5 Grouped permutation importance (Sep–Oct, not test)

Each family is shuffled **together** with one permutation (5 repeats). Correlated features within a family can't stand in for each other, and the history features *are* strongly correlated (notebook 01, §8). This is descriptive, not a selection criterion.

In [11]:
perm = pd.read_csv(R / "permutation_importance_groups.csv")
display(perm.pivot(index="group", columns="model", values="pr_auc_drop").round(4))
gain = pd.read_csv(R / "xgb_gain_importance.csv", index_col=0)["gain"]
psi = pd.read_csv(R / "psi.csv")
figs.update(pipeline.evaluation_figures(labels, final_preds, choice, pipeline.monthly_drift(labels, final_preds, choice), gain, perm, abl, psi))
figs["perm_importance_groups"]

model,LogReg,XGBoost
group,,
categorical,0.0376,0.0147
congestion,0.0464,0.0150
history_day,0.0261,0.0089
history_entity,0.0287,0.0178
schedule,0.0423,0.0537


<Figure size 550x220 with 1 Axes>

* **Grouped permutation** (the primary measure): for the **final model (LogReg)**, influence ranks congestion (0.046), schedule (0.042), categorical (0.038), history_entity (0.029), history_day (0.026). For XGBoost, the ranking is schedule (0.054), history_entity (0.018), congestion (0.015), categorical (0.015), history_day (0.009). Every family contributes, which agrees with the LogReg ablation in A.4.
* **LogReg coefficients are conditional and collinear.** The largest pair is `hist_logn_carrier_14d` and `hist_logn_carrier_7d`, of opposite sign on nearly collinear volumes, so together they encode the *change* in a carrier's recent volume and neither is interpretable alone. This is why grouped permutation, not coefficients, is the importance measure.

In [12]:
pd.read_csv(R / "logreg_coefficients.csv", index_col=0).head(10)

,coef
num__hist_logn_carrier_14d,0.6607
num__hist_logn_carrier_7d,-0.6349
cat__dest_BOS,-0.5846
cat__dest_PHL,0.4692
cat__origin_EWR,-0.4503
cat__dest_LAX,-0.4051
cat__dest_SNA,-0.3870
cat__dest_ORD,-0.3384
cat__carrier_DL,-0.3182
cat__dest_PSE,-0.3115


# Part B · Exploratory test-set findings (ablation test diffs, calibration study, target sensitivity)

*Test = Nov–Dec 2013. Nothing here chose anything: the model, the calibration rule, the policies and N = 28 were all fixed beforehand (Part A, [§0](#protocol)). Because test numbers were visible during development (§0), small differences are exploratory, not confirmatory.*

**B.1** reports the pre-specified rules on test. **B.2** is the post-hoc analysis (calibration study, ablation test differences, target sensitivity, covariate shift).

## B.1 Pre-specified rules evaluated on test

### B.1.1 Headline test metrics: baselines, calibration variants and the final model

In [13]:
thresholds = pipeline.frozen_thresholds(labels, final_preds)
results = pipeline.metrics_table(labels, final_preds, thresholds, choice)
print(artefacts.assert_matches(results, "metrics_all.csv"))
headline = pipeline.headline_table(results)
print(artefacts.assert_matches(headline, "headline_test.csv"))
headline[["model", "calibration", *pipeline.SHOW_COLS]]

OK: identical to results/metrics_all.csv
OK: identical to results/headline_test.csv


,model,calibration,pr_auc,roc_auc,brier,ece,ece_equal_mass,mean_pred,base_rate,oracle_recall@p0.30,oracle_recall@p0.40,oracle_recall@p0.50
0,Prior (base rate),none,0.2583,0.5000,0.1916,0.0066,0.0842,0.2649,0.2583,0.0000,0.0000,0.0000
1,Route history,none (smoothed rate),0.3534,0.6208,0.1869,0.0373,0.0380,0.2420,0.2583,0.7868,0.2444,0.0163
2,LogReg (raw),raw,0.3937,0.6537,0.1811,0.0190,0.0216,0.2581,0.2583,0.8469,0.4303,0.1294
3,"LogReg (Platt, rule-selected)",Platt (rule-selected),0.3937,0.6537,0.1872,0.0815,0.0815,0.1768,0.2583,0.8469,0.4303,0.1294
4,LogReg (rolling Platt 28d),rolling Platt 28d (causal),0.3918,0.6575,0.1809,0.0269,0.0265,0.2318,0.2583,0.8704,0.4320,0.1203
5,"XGBoost (raw, rule-selected)",raw (rule-selected),0.4070,0.6619,0.1797,0.0267,0.0266,0.2317,0.2583,0.8633,0.4740,0.1613
6,XGBoost (rolling Platt 28d),rolling Platt 28d (causal),0.3960,0.6595,0.1803,0.0241,0.0231,0.2444,0.2583,0.8740,0.4159,0.1218


**Reading the table.** *Prior* and *Route history* are the reference rows; every learned row names its calibration variant. `LogReg (Platt, rule-selected)` is the model as the pre-declared rule deploys it. Its ranking (PR-AUC, ROC-AUC) is identical to `LogReg (raw)`, since Platt is monotone, but its calibration is not:

* **LogReg (Platt, rule-selected): test ECE 0.082** vs **LogReg (raw): 0.019** vs **LogReg (rolling Platt 28d): 0.027**. The static rule makes LogReg's calibration *worse* on test, and the next section shows where: it **fails in December**. Raw scores happen to be well calibrated on Nov–Dec overall only because the training base rate (26.5%) is close to the test base rate (25.8%); that is a coincidence of this split. The causal rolling Platt tracks the shift.
* **PR-AUC:** LogReg (final) 0.394, XGBoost (raw, rule-selected) 0.407, Route history 0.353, Prior 0.258 (= the base rate). Both learned models beat both references.
* **The prior's low ECE (0.007) is not skill:** a constant at the training base rate is calibrated on average when the test base rate is close. Equal-mass ECE (0.084) and PR-AUC expose it.
* **`oracle_recall@p`** tunes the threshold on the test labels; it is for comparing models only (§1).

### B.1.2 November and December separately

In [14]:
by_month = pipeline.headline_by_month(results)
print(artefacts.assert_matches(by_month, "headline_by_month.csv"))
by_month[["model", "calibration", "split", "base_rate", "pr_auc", "roc_auc", "brier", "ece", "mean_pred"]]

OK: identical to results/headline_by_month.csv


,model,calibration,split,base_rate,pr_auc,roc_auc,brier,ece,mean_pred
0,Prior (base rate),none,test_m11,0.1810,0.1810,0.5000,0.1553,0.0839,0.2649
1,Route history,none (smoothed rate),test_m11,0.1810,0.2208,0.5596,0.1492,0.0352,0.1750
2,LogReg (raw),raw,test_m11,0.1810,0.2913,0.6552,0.1454,0.0553,0.2349
3,"LogReg (Platt, rule-selected)",Platt (rule-selected),test_m11,0.1810,0.2913,0.6552,0.1418,0.0208,0.1602
4,LogReg (rolling Platt 28d),rolling Platt 28d (causal),test_m11,0.1810,0.2854,0.6500,0.1419,0.0065,0.1772
5,"XGBoost (raw, rule-selected)",raw (rule-selected),test_m11,0.1810,0.2700,0.6382,0.1435,0.0191,0.1922
6,XGBoost (rolling Platt 28d),rolling Platt 28d (causal),test_m11,0.1810,0.2607,0.6311,0.1438,0.0130,0.1761
7,Prior (base rate),none,test_m12,0.3355,0.3355,0.5000,0.2279,0.0706,0.2649
8,Route history,none (smoothed rate),test_m12,0.3355,0.3944,0.5740,0.2246,0.0602,0.3088
9,LogReg (raw),raw,test_m12,0.3355,0.4555,0.6315,0.2168,0.0595,0.2812


The static Platt calibration (fit on calm Sep–Oct) scales risk *down*: right for November (ECE 0.021), badly wrong for December (0.142, December base rate 33.5%). Raw scores and rolling Platt are far less sensitive to the month. **The static rule fails in December.**

### B.1.3 LogReg vs XGBoost: day-block bootstrap (raw-ranking metrics, rule-selected variants)

In [15]:
boot = pipeline.bootstrap_compare(labels, final_preds)
assert json.loads(json.dumps(boot)) == S["bootstrap_test"], "bootstrap differs from summary.json"
pd.DataFrame({k: {"XGBoost 95% CI": np.round(v["xgboost_ci95"], 3), "LogReg 95% CI": np.round(v["logreg_ci95"], 3),
                  "XGB − LR 95% CI": np.round(v["diff_xgboost_minus_logreg_ci95"], 3),
                  "P(XGB not better)": v["p_xgboost_not_better"]} for k, v in boot.items()}).T

,XGBoost 95% CI,LogReg 95% CI,XGB − LR 95% CI,P(XGB not better)
pr_auc,"[0.348, 0.465]","[0.34, 0.45]","[-0.01, 0.041]",0.156
roc_auc,"[0.633, 0.691]","[0.62, 0.682]","[-0.012, 0.029]",0.208


On test, XGBoost (raw, rule-selected) is ahead of LogReg (Platt, rule-selected) by +0.013 PR-AUC, but the **95% CI includes 0** ([-0.010, +0.041]), so the two are statistically indistinguishable, in line with the CV result in A.1. The honest conclusion: **hand-built day-ahead summaries leave little interaction structure for trees to exploit**. That's the gap the deep-learning phase targets. The final-model choice stands on the pre-declared CV rule (A.1), not on this comparison.

### B.1.4 PR / ROC curves and reliability (references included)

In [16]:
for name, fig in figs.items():
    plots.save(fig, name)
figs["pr_roc_test"]

<Figure size 850x360 with 2 Axes>

In [17]:
figs["reliability_logreg_test"]

<Figure size 440x480 with 2 Axes>

In [18]:
figs["reliability_xgboost_test"]

<Figure size 440x480 with 2 Axes>

The dashed grey lines are the references: the prior (horizontal base-rate line in PR, the diagonal in ROC) and, in the reliability plots, the route-history heuristic. The reliability plots show the raw, static-Platt and rolling-Platt variants of each model.

### B.1.5 Operating policies (fixed before test)

* **PRIMARY: daily capacity policy.** Each day, flag the **10%** highest-risk flights, as an operations desk with fixed review capacity would. It uses only the ranking, so it is immune to base-rate shift, and a monotone recalibration within a day (raw, Platt, rolling Platt) cannot change it. The 10% share is an **a-priori capacity assumption** and was not tuned. Exact ties are broken at random with a fixed seed.
* **SECONDARY: frozen threshold.** On Sep–Oct, take the lowest threshold that reaches precision ≥ 0.40, and keep it fixed for Nov–Dec. The target was **0.50** first, but that is unreachable on Sep–Oct, so it was lowered to 0.40 (a Sep–Oct decision). The threshold is on the *calibrated* scale of the row's variant.

In [19]:
policies = pipeline.operating_policies(labels, final_preds, thresholds, choice)
print(artefacts.assert_matches(policies, "operating_policies.csv"))
policies[["model", "calibration", "split", "role", "policy", "flag_rate", "precision", "recall", "lift", "threshold"]].round(3)

OK: identical to results/operating_policies.csv


,model,calibration,split,role,policy,flag_rate,precision,recall,lift,threshold
0,Prior (base rate),none,calib,primary,daily top 10%,0.100,0.153,0.098,0.977,NaN
1,Prior (base rate),none,calib,secondary,frozen thr for P>=0.4 (from Sep-Oct),0.000,NaN,0.000,NaN,inf
2,Prior (base rate),none,test,primary,daily top 10%,0.101,0.243,0.095,0.941,NaN
3,Prior (base rate),none,test,secondary,frozen thr for P>=0.4 (from Sep-Oct),0.000,NaN,0.000,NaN,inf
4,Route history,none (smoothed rate),calib,primary,daily top 10%,0.100,0.232,0.148,1.476,NaN
5,Route history,none (smoothed rate),calib,secondary,frozen thr for P>=0.4 (from Sep-Oct),0.001,0.417,0.002,2.655,0.469
6,Route history,none (smoothed rate),test,primary,daily top 10%,0.101,0.358,0.139,1.387,NaN
7,Route history,none (smoothed rate),test,secondary,frozen thr for P>=0.4 (from Sep-Oct),0.059,0.425,0.097,1.646,0.469
8,"LogReg (Platt, rule-selected)",Platt (rule-selected),calib,primary,daily top 10%,0.100,0.303,0.194,1.932,NaN
9,"LogReg (Platt, rule-selected)",Platt (rule-selected),calib,secondary,frozen thr for P>=0.4 (from Sep-Oct),0.038,0.400,0.097,2.549,0.331


### Decision support: rescheduling / add-buffer candidates

Each test day, the final model's ranking selects the top-k% flights as candidates for a capacity-limited review desk, k in {5, 10, 15, 20}, against the prior (random) and route-history rankings.

**This shows ranking quality, not realised operational savings**: the data contain no cost or schedule-change model.

In [20]:
resched = pipeline.rescheduling_candidates(labels, final_preds, choice)
print(artefacts.assert_matches(resched, "rescheduling_candidates.csv"))
resched[["model", "calibration", "top_share", "flagged_per_day", "precision", "recall", "lift", "delayed_per_100_flagged"]].round(3)

OK: identical to results/rescheduling_candidates.csv


,model,calibration,top_share,flagged_per_day,precision,recall,lift,delayed_per_100_flagged
0,Prior (base rate),none,0.05,44.705,0.251,0.049,0.971,25.083
1,Prior (base rate),none,0.10,88.984,0.243,0.095,0.941,24.300
2,Prior (base rate),none,0.15,133.148,0.250,0.145,0.967,24.982
3,Prior (base rate),none,0.20,177.393,0.252,0.195,0.974,25.164
4,Route history,none (smoothed rate),0.05,44.705,0.386,0.076,1.495,38.614
5,Route history,none (smoothed rate),0.10,88.984,0.358,0.139,1.387,35.814
6,Route history,none (smoothed rate),0.15,133.148,0.346,0.201,1.339,34.597
7,Route history,none (smoothed rate),0.20,177.393,0.339,0.263,1.313,33.916
8,"LogReg (Platt, rule-selected)",Platt (rule-selected),0.05,44.705,0.469,0.092,1.817,46.938
9,"LogReg (Platt, rule-selected)",Platt (rule-selected),0.10,88.984,0.444,0.173,1.720,44.436


In [21]:
# the primary policy only depends on within-day ranks: raw / Platt / rolling Platt flag (almost) the same flights
flag = {k: evaluate.flag_daily_top_share(labels.dates["test"], final_preds[k]["test"])
        for k in ["logreg_raw", "logreg_rule_selected", f"logreg_rolling{config.ROLLING_CALIB_DAYS}"]}
pd.Series({f"{a} vs {b}": float((flag[a] == flag[b]).mean()) for a, b in
           [("logreg_raw", "logreg_rule_selected"), ("logreg_raw", f"logreg_rolling{config.ROLLING_CALIB_DAYS}")]},
          name="share of flights flagged identically").to_frame()

,share of flights flagged identically
logreg_raw vs logreg_rule_selected,1.0
logreg_raw vs logreg_rolling28,1.0


**Primary result (test): the daily top-10% policy applied to the final model, `LogReg (Platt, rule-selected)`, flags 10% of flights with 0.444 precision and 0.173 recall, a 1.72× lift over random flagging.** The references: the prior (random flagging) gives lift 0.94 and the route-history heuristic 1.39. XGBoost (raw, rule-selected) gets lift 1.68. Precision is stable in relative terms between Sep–Oct (0.303, lift 1.93) and Nov–Dec; the absolute precision rises with the base rate.

The **frozen threshold** behaves unpredictably under the shift. For LogReg (Platt, rule-selected) it flags 3.8% of flights in Sep–Oct but 7.5% in Nov–Dec, because the base rate (and therefore the score distribution) moved. The prior never reaches precision 0.40, so its threshold is infinite and it flags nothing. A fixed probability cut-off doesn't carry over between seasons.

## B.2 Exploratory (post-hoc) findings

### B.2.1 Calibration study: every variant on test, Nov and Dec

In [22]:
study = pipeline.calibration_study(results)
print(artefacts.assert_matches(study, "calibration_study.csv"))
study[["model", "calibration", "ece_test", "ece_test_m11", "ece_test_m12", "ece_equal_mass_test", "brier_test", "brier_test_m11", "brier_test_m12", "mean_pred_test"]].round(4)

OK: identical to results/calibration_study.csv


,model,calibration,ece_test,ece_test_m11,ece_test_m12,ece_equal_mass_test,brier_test,brier_test_m11,brier_test_m12,mean_pred_test
0,Prior (base rate),none,0.0066,0.0839,0.0706,0.0842,0.1916,0.1553,0.2279,0.2649
1,Route history,none (smoothed rate),0.0373,0.0352,0.0602,0.0380,0.1869,0.1492,0.2246,0.2420
2,LogReg (raw),raw,0.0190,0.0553,0.0595,0.0216,0.1811,0.1454,0.2168,0.2581
3,"LogReg (Platt, static)","Platt (static, fit Sep-Oct)",0.0815,0.0208,0.1421,0.0815,0.1872,0.1418,0.2326,0.1768
4,"LogReg (isotonic, static)","isotonic (static, fit Sep-Oct)",0.0817,0.0226,0.1416,0.0802,0.1873,0.1419,0.2327,0.1781
5,LogReg (rolling Platt 14d),rolling Platt 14d (causal),0.0215,0.0110,0.0385,0.0230,0.1807,0.1431,0.2181,0.2512
6,LogReg (rolling Platt 28d),rolling Platt 28d (causal),0.0269,0.0065,0.0499,0.0265,0.1809,0.1419,0.2199,0.2318
7,LogReg (rolling Platt 56d),rolling Platt 56d (causal),0.0510,0.0135,0.0886,0.0510,0.1820,0.1416,0.2223,0.2073
8,XGBoost (raw),raw,0.0267,0.0191,0.0642,0.0266,0.1797,0.1435,0.2158,0.2317
9,"XGBoost (Platt, static)","Platt (static, fit Sep-Oct)",0.0812,0.0329,0.1309,0.0812,0.1864,0.1440,0.2286,0.1771


**Every calibration variant on test (Nov–Dec), with Nov and Dec separately, and the references first:**
* **Static recalibration fails under the shift.** Platt and isotonic were fitted on calm Sep–Oct (15.7% late), so they scale risk *down*. That's correct for November (LogReg Platt ECE 0.021) and badly wrong for December (0.142, base rate 33.5%). The rule was applied as fixed, and its test result is reported as is.
* **Raw scores** are well calibrated on Nov–Dec overall (LogReg ECE 0.019), only because the training base rate is close to the test base rate. That's a coincidence of this split, not something you could count on in deployment.
* **Causal rolling Platt** (N = 28, fixed a priori) **adapts**: LogReg test ECE 0.027 (Nov 0.007, Dec 0.05), XGBoost 0.024. Shorter windows adapt faster (N = 14: 0.021 for LogReg) and longer ones lag (N = 56: 0.051). Those two were not selected; they're reported only.
* **Lesson:** post-hoc calibration (Guo et al.) assumes the calibration data look like deployment data. Under temporal prior shift (Ovadia et al., 2019) a frozen calibrator breaks, and the remedy is to **recalibrate continuously on recent known outcomes**.

### B.2.2 Monthly calibration drift

In [23]:
drift = pipeline.monthly_drift(labels, final_preds, choice)
print(artefacts.assert_matches(drift, "monthly_calibration_drift.csv"))
figs["monthly_drift_final"]

OK: identical to results/monthly_calibration_drift.csv


<Figure size 650x320 with 1 Axes>

In [24]:
drift.pivot_table(index="month", columns="model", values=["observed", "predicted"]).round(3)

observed                                                                    predicted                                                      
model LogReg (Platt, static) LogReg (raw) LogReg (rolling Platt 28d) XGBoost (raw) LogReg (Platt, static) LogReg (raw) LogReg (rolling Platt 28d) XGBoost (raw)
month                                                                                                                                                          
1                      0.238        0.238                      0.238         0.238                  0.142        0.207                        NaN         0.236
2                      0.238        0.238                      0.238         0.238                  0.174        0.254                        NaN         0.258
3                      0.236        0.236                      0.236         0.236                  0.173        0.253                        NaN         0.249
4                      0.288        0.288                      0.288         0.288                  0.179        0.262                        NaN         0.268
5                      0.227        0.227                      0.227         0.227                  0.182        0.265                        NaN         0.236
6                      0.315        0.315                      0.315         0.315                  0.190        0.277                        NaN         0.286
7                      0.322        0.322                      0.322         0.322                  0.215        0.308                        NaN         0.319
8                      0.246        0.246                      0.246         0.246                  0.189        0.276                        NaN         0.257
9                      0.137        0.137                      0.137         0.137                  0.155        0.227                      0.159         0.210
10                     0.176        0.176                      0.176         0.176                  0.159        0.233                      0.154         0.196
11                     0.181        0.181                      0.181         0.181                  0.160        0.235                      0.177         0.192
12                     0.335        0.335                      0.335         0.335                  0.193        0.281                      0.286         0.271

The raw LogReg predictions follow the **shape** of the season but **compress it toward the middle**. They over-predict in calm Sep–Oct and under-predict December's 33.5%. The rolling-Platt column shows the correction tracking each month.

### B.2.3 Ablation: test differences (confirmation only; claims rest on A.4)

In [25]:
abl = pd.read_csv(R / "ablation.csv")
display(abl[["model", "calibration", "features", "test_pr_auc", "test_roc_auc"]].round(4))
steps[["model", "calibration", "step", "test_diff", "test_diff_ci_lo", "test_diff_ci_hi"]].round(4)

,model,calibration,features,test_pr_auc,test_roc_auc
0,Prior (base rate),none,(baseline),0.2583,0.5000
1,Route history,none (smoothed rate),(baseline),0.3534,0.6208
2,LogReg,raw,schedule + ids,0.3450,0.6126
3,LogReg,raw,+ congestion,0.3522,0.6145
4,LogReg,raw,+ history_entity,0.3846,0.6428
5,LogReg,raw,+ history_day,0.3937,0.6537
6,LogReg,raw,history only,0.3903,0.6566
7,XGBoost,raw,schedule + ids,0.3430,0.6193
8,XGBoost,raw,+ congestion,0.3454,0.6186
9,XGBoost,raw,+ history_entity,0.3973,0.6536


,model,calibration,step,test_diff,test_diff_ci_lo,test_diff_ci_hi
0,LogReg,raw,+ congestion,0.0072,-0.0092,0.0275
1,LogReg,raw,+ history_entity,0.0324,0.0154,0.0493
2,LogReg,raw,+ history_day,0.0091,-0.0091,0.0268
3,XGBoost,raw,+ congestion,0.0024,-0.0101,0.0147
4,XGBoost,raw,+ history_entity,0.0520,0.0263,0.0772
5,XGBoost,raw,+ history_day,0.0096,-0.0238,0.0454


The test difference of each step carries a paired day-block bootstrap 95% CI. **Final model (LogReg, raw):** congestion +0.007 [-0.009, +0.027] (CI includes 0); entity history +0.032 [+0.015, +0.049] (CI excludes 0); day-level history +0.009 [-0.009, +0.027] (CI includes 0). **XGBoost (raw):** congestion +0.002 [-0.010, +0.015] (CI includes 0); entity history +0.052 [+0.026, +0.077] (CI excludes 0); day-level history +0.010 [-0.024, +0.045] (CI includes 0). History alone (LogReg CV 0.4478) is weaker than schedule + history, so the families complement each other. The two reference models bracket the table: the prior (CV 0.2775) and route history (CV 0.3476) sit below every learned feature set.

### B.2.4 Target sensitivity: counting cancelled and diverted flights as delayed

In [26]:
sens = pd.read_csv(R / "target_sensitivity.csv")
sens.round(4)

,target,model,calibration,n_test,base_rate,pr_auc,roc_auc,brier,ece,top10_precision,top10_recall,top10_lift,xgb_minus_lr_pr_auc_ci
0,arr_delay>=15 (main),Prior (base rate),none,53991,0.2583,0.2583,0.5000,0.1916,0.0066,0.2430,0.0946,0.9408,NaN
1,arr_delay>=15 (main),Route history,none (smoothed rate),53991,0.2583,0.3534,0.6208,0.1869,0.0373,0.3581,0.1394,1.3865,NaN
2,arr_delay>=15 (main),LogReg (raw),raw,53991,0.2583,0.3937,0.6537,0.1811,0.0190,0.4444,0.1730,1.7203,"[-0.010, +0.041]"
3,arr_delay>=15 (main),XGBoost (raw),raw,53991,0.2583,0.4070,0.6619,0.1797,0.0267,0.4346,0.1692,1.6825,"[-0.010, +0.041]"
4,arr_delay>=15 OR cancelled/diverted,Prior (base rate),none,55403,0.2772,0.2772,0.5000,0.2005,0.0116,0.2622,0.0951,0.9459,NaN
5,arr_delay>=15 OR cancelled/diverted,Route history,none (smoothed rate),55403,0.2772,0.3739,0.6215,0.1958,0.0472,0.3768,0.1366,1.3593,NaN
6,arr_delay>=15 OR cancelled/diverted,LogReg (raw),raw,55403,0.2772,0.4461,0.6609,0.1864,0.0240,0.4784,0.1735,1.7260,"[-0.013, +0.037]"
7,arr_delay>=15 OR cancelled/diverted,XGBoost (raw),raw,55403,0.2772,0.4549,0.6694,0.1848,0.0290,0.4754,0.1724,1.7150,"[-0.013, +0.037]"


Counting cancelled and diverted flights as delayed (a passenger-disruption view) raises the base rate from 25.8% to 27.7%. PR-AUC rises with it, mainly because PR-AUC's baseline is the base rate (the prior row moves in step). **The conclusions don't change:**
* XGBoost (raw) and LogReg (raw) remain indistinguishable (the CI of the difference includes 0, [-0.013, +0.037]);
* the primary policy gives the same lift (LogReg 1.73, XGBoost 1.71; prior 0.95);
* both learned models still beat both references.

The findings don't depend on how cancellations are handled.

### B.2.5 Covariate shift: PSI and adversarial validation

In [27]:
display(psi.head(12).round(3))
figs["psi_top15"]

,feature,group,psi_calib,psi_test
0,days_to_holiday,schedule,0.261,4.769
1,hist_logn_origin_7d,history_entity,2.269,1.265
2,hist_rate_network_7d,history_day,5.147,1.058
3,hist_rate_origin_7d,history_entity,3.433,0.755
4,sched_origin_day,congestion,2.056,0.610
5,hist_rate_carrier_14d,history_entity,2.631,0.383
6,hist_rate_dest_14d,history_entity,2.143,0.292
7,hist_logn_carrier_14d,history_entity,0.472,0.289
8,hist_cancel_network_1d,history_day,1.229,0.222
9,hist_rate_carrier_7d,history_entity,1.699,0.209


<Figure size 550x500 with 1 Axes>

In [28]:
pd.read_csv(R / "adversarial_validation.csv").round(3)

,comparison,features,n_features,adversarial_auc,top_features
0,train vs calib,all features,45,1.000,"hist_rate_network_7d, hist_rate_dest_14d, hist_rate_carrier_14d, hist_logn_origin_7d"
1,train vs calib,categorical,3,0.562,"dest, carrier, origin"
2,train vs calib,schedule,10,0.785,"days_to_holiday, dow_cos, dow_sin, arr_tod_sin"
3,train vs calib,congestion,7,0.983,"sched_origin_day, sched_carrier_origin_day, sched_origin_hour, sched_origin_3h"
4,train vs calib,history_entity,18,1.000,"hist_rate_origin_7d, hist_rate_dest_14d, hist_rate_carrier_14d, hist_logn_origin_7d"
5,train vs calib,history_day,7,1.000,"hist_rate_network_7d, hist_rate_network_1d, hist_logn_origin_1d, hist_cancel_network_1d"
6,train vs test,all features,45,1.000,"days_to_holiday, origin, hist_logn_origin_7d, hist_rate_network_7d"
7,train vs test,categorical,3,0.562,"dest, carrier, origin"
8,train vs test,schedule,10,0.921,"days_to_holiday, is_weekend, dow_cos, dow_sin"
9,train vs test,congestion,7,0.963,"sched_origin_day, sched_carrier_origin_day, sched_origin_3h, tail_legs_day"


* **PSI** (train as reference; above 0.25 counts as a large shift): the largest shifts into Sep–Oct are hist_rate_network_7d, hist_rate_origin_7d, hist_rate_carrier_14d (history rates follow the season by design), and into Nov–Dec days_to_holiday, hist_logn_origin_7d, hist_rate_network_7d; `days_to_holiday` is the largest (PSI 4.8) because Thanksgiving, Christmas and New Year all fall in the test window.
* **Adversarial validation:** a classifier separates training rows from calibration or test rows nearly **perfectly using all features**, and also using **history features alone**. Day-level history takes one value per day, so it identifies the period trivially. That shows *why* the day-level group is a memorisation risk (notebook 02, §8).
* The per-family rows show where the shift lives: **categorical** 0.56 (vs calib) / 0.56 (vs test); **schedule** 0.78 (vs calib) / 0.92 (vs test); **congestion** 0.98 (vs calib) / 0.96 (vs test); **history_entity** 1.00 (vs calib) / 1.00 (vs test); **history_day** 1.00 (vs calib) / 1.00 (vs test). The IDs (carrier, origin, destination) are nearly stable; schedule and congestion shift through season and holiday proximity; history separates the periods completely by design.
* This is a strong, measured covariate shift, and it is the root cause of the calibration findings in B.2.1.

### B.2.6 XGBoost gain (top 20)

In [29]:
figs["xgb_gain_top20"]

<Figure size 550x640 with 1 Axes>

## Findings

1. **Leak fixed, and it was material.** 31% of D-1 arrivals land after the 20:00 decision time, and they're twice as often late (38% vs 18%). Counting them (24:00 clock) would score 0.4812 vs 0.4705 in rolling CV: a leak worth about 0.011 PR-AUC. A 2-day lag would cost 0.012 (CV).
2. **Final model: LogReg (Platt, rule-selected)** by the CV rule declared before test (CV as deployed, 0.4705 vs 0.4450). LogReg and XGBoost are **not statistically distinguished** (CV fold sd about 0.07; test bootstrap CI includes 0), so the choice rests on the rule, not on superiority. Day-ahead signal is weak and mostly additive. Both beat the prior and route-history references.
3. **Every feature family adds CV value for the final model.** Month features don't help (CV).
4. **Calibration:** the static rule fixed in advance adopts Platt for LogReg (test ECE 0.082, vs 0.019 raw), which **fails in December** under the seasonal shift. **Causal rolling recalibration** (N = 28, a priori; LogReg ECE 0.027, XGBoost 0.024) tracks it.
5. **Primary operating policy (daily top 10%, LogReg Platt rule-selected): precision 0.444, recall 0.173, lift 1.72** on test, against lift 0.94 for random flagging. A frozen probability threshold doesn't transfer between seasons.
6. **Robust to the target definition** (cancellations as delays), and the covariate-shift diagnostics explain the calibration behaviour.

## Limitations and assumptions
* **Single year, three origins.** Each season and airport effect is observed once. The **chronological split is retained** (train on the past, test on the future), but a single year (2013) and three NYC origins **cannot establish cross-year or cross-airport generalisation**; no out-of-year validation is possible.
* **No weather data, by design.** Observed weather for day D would **leak at the 20:00 D-1 decision clock**: it is not known then. Weather could only be added from **archived forecasts issued before 20:00 on D-1** (e.g. archived NWS/TAF forecasts), never from observations. It is the main missing driver of day-to-day delays, and a stated gap, not a bug.
* **Only NYC departures**, so aircraft rotation is only partly visible.
* **Assumptions:** tail-number assignment known by 20:00 on D-1; US *federal* holiday calendar; cancellations known by the scheduled departure time; destination time zones from a fixed table plus the 2013 DST rule; outcomes known at actual gate arrival.
* **Test was not perfectly clean** (§0). Small test differences should be read with caution.